In [ ]:
import os
import geopandas as gpd
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import gdown
from scipy.stats import linregress
import cmcrameri.cm as cmc
from matplotlib.colors import LightSource

# Create data directory
data_folder = "data"
os.makedirs(data_folder, exist_ok=True)

# Dictionary of file IDs and their output names
datasets = {
    "scf_alps_25yr.tif": "1Uds7kAGnbl7gdFQrDRIdeRXMSpztiO3i",
    "dem_alps.tif": "1Q895LhVvtbEoeg1LL9wQFGdmg11M9x_N",
    "lakes_alps.tif": "1VPuhMBWuX_JX3Ud798vd9gccPUAwqmIz",
    "alpine_perimeter.zip": "1945H8BAefR6C8umzaLKPhrVfGT_uFRNN",
    "bahnen-winter_2056.gpkg": "1c2f4Nwe5f4Pp5leDSuQZj1nyRoluJoOT",
}

for filename, file_id in datasets.items():
    filepath = os.path.join(data_folder, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, filepath, quiet=False)
    else:
        print(f"{filename} already exists.")

scf_fp = os.path.join(data_folder, "scf_alps_25yr.tif")
dem_fp = os.path.join(data_folder, "dem_alps.tif")
lakes_fp = os.path.join(data_folder, "lakes_alps.tif")
perim_fp = os.path.join(data_folder, "alpine_perimeter.zip")
lifts_fp = os.path.join(data_folder, "bahnen-winter_2056.gpkg")

In [ ]:
# 1. Load and process vector data
perimeter = gpd.read_file(perim_fp)

lifts = gpd.read_file(lifts_fp)
chairlifts = lifts[lifts['type'] == 'chairlift'].copy()
chairlifts_pts = chairlifts.copy()
chairlifts_pts['geometry'] = chairlifts.geometry.centroid

# Reproject to EPSG:3035
chairlifts_pts = chairlifts_pts.to_crs(epsg=3035)

# 2. Load and clip raster data using rioxarray
bounds = perimeter.total_bounds
scf_raw = rioxarray.open_rasterio(scf_fp).rio.clip_box(*bounds)
dem_raw = rioxarray.open_rasterio(dem_fp).rio.clip_box(*bounds)
lakes_raw = rioxarray.open_rasterio(lakes_fp).rio.clip_box(*bounds)

# 3. Format the data cube's time dimension
scf_cube = scf_raw.rename({'band': 'time'})
scf_cube = scf_cube.assign_coords(time=np.arange(2001, 2026))

print(f"Cube Shape: {scf_cube.shape} (Years, Y, X)")

# 4. Blend Mean SCF on Hillshade
ls = LightSource(azdeg=315, altdeg=45)
scf_mean = scf_cube.mean(dim='time').squeeze().values
dem_data = dem_raw.values[0]
hillshade = ls.hillshade(dem_data, vert_exag=1)
extent = [dem_raw.x.min(), dem_raw.x.max(), dem_raw.y.min(), dem_raw.y.max()]

# Colormap
cmap = cmc.batlowW_r
norm = plt.Normalize(vmin=np.nanmin(scf_mean), vmax=np.nanmax(scf_mean))

# Map SCF to RGB
scf_rgb = cmap(norm(scf_mean))[:, :, :3]

# Shade the RGB
blended_rgb = ls.shade_rgb(scf_rgb, elevation=hillshade, blend_mode='overlay', vert_exag=2)

# Add Lakes
lakes_mask = lakes_raw.values[0] > 0
lake_color = np.array([0.0, 0.6, 0.8])
blended_rgb[lakes_mask] = lake_color

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(blended_rgb, extent=extent)

# Plot Chairlifts
chairlifts_pts.plot(ax=ax, color='red', marker='.', markersize=2, label='Chairlifts', zorder=5)

# Plot Perimeter
perimeter.boundary.plot(ax=ax, color='black', linewidth=1, alpha=0.5, label='Alpine Perimeter', zorder=4)

# Add Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.6, label='Mean Snow Cover Fraction')

plt.title('Mean Snow Cover Fraction (25yr) with Topography')
plt.xlabel('Easting (m)')
plt.ylabel('Northing (m)')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# 1. Define Coordinates: Set up the coordinates for Zermatt in EPSG:3035
zermatt_x = 4146515
zermatt_y = 2547946

# 2. Extract Time Series: Drill down through the 25 layers of the scf_cube
# for the pixel nearest to the Zermatt coordinates.
zermatt_scf_cube = scf_cube.sel(x=zermatt_x, y=zermatt_y, method="nearest")

# 3. Trend Analysis: Calculate Linear trend using polyfit
scf_cube_fit = zermatt_scf_cube.polyfit(dim="time", deg=1)
scf_cube_slope = scf_cube_fit.polyfit_coefficients.sel(degree=1)

# Convert annual rate to a decadal trend (multiply by 10)
scf_cube_trend_line = xr.polyval(zermatt_scf_cube.time, scf_cube_fit.polyfit_coefficients)

print(f"SCF trend at Zermatt: {float(scf_cube_slope):.4f} SCF per year")
print(f"SCF trend at Zermatt: {float(scf_cube_slope * 10):.4f} SCF per decade")

# 4. Visualize: Plot the Snow Cover Frequency over the 25-year period
plt.figure(figsize=(10, 4))
zermatt_scf_cube.plot(marker="o", label="Observed SCF")
scf_cube_trend_line.plot(color="red", linewidth=2, label="Linear trend")

# Add appropriate titles and labels
plt.title("Snow Cover Fraction (SCF) Time Series and Trend near Zermatt")
plt.ylabel("Snow Cover Fraction")
plt.xlabel("Year")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
# 1. Apply linear regression across the time dimension for the entire cube
trends = scf_cube.polyfit(dim="time", deg=1)

# 2. Extract and scale slope (degree=1 is change per year, * 10 for change per decade)
trend_map = trends.polyfit_coefficients.sel(degree=1) * 10

# 3. Apply Lake Mask (keep land where lakes_raw is 0)
# We squeeze lakes_raw to match the 2D dimensions of the trend_map
lake_mask = lakes_raw.squeeze()
trend_map_masked = trend_map.where(lake_mask == 0)

# 4. Plot the regional trend map
fig, ax = plt.subplots(figsize=(12, 6))

# Use RdBu_r so red indicates a negative trend (snow loss)
trend_map_masked.plot(
    ax=ax, 
    cmap='RdBu', 
    vmin=-0.08,
    vmax=0.08,
    cbar_kwargs={'label': 'SCF Change per Decade'}
)

# Overlay Perimeter
perimeter.boundary.plot(
    ax=ax, 
    color="black", 
    linewidth=1, 
    alpha=0.6,
    label="Alpine Perimeter"
)

ax.set_title("Decadal Trend in Snow Cover Frequency (2001-2025)")
plt.legend(loc='lower right')
plt.xlabel('Easting (m)')
plt.ylabel('Northing (m)')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import hvplot.pandas

# 1. Load and filter using existing file paths
lifts = gpd.read_file(lifts_fp)
chairlifts = lifts[lifts['type'] == 'chairlift'].copy()

# 2. Extract centroids and reproject to EPSG:3035 to match rasters
chairlifts['geometry'] = chairlifts.geometry.centroid
chairlifts_pts = chairlifts.to_crs("EPSG:3035")

# 3. Sample raster data at points using corrected variable names
x_coords = chairlifts_pts.geometry.x.values
y_coords = chairlifts_pts.geometry.y.values

# Initialize empty lists to store sampled values
scf_slopes = []
elevations = []

# Loop through coordinates to sample from the trend map and DEM
for x, y in zip(x_coords, y_coords):
    scf_slopes.append(trend_map_masked.sel(x=x, y=y, method='nearest').values.item())
    elevations.append(dem_raw.sel(x=x, y=y, method='nearest').values.item())

chairlifts_pts['scf_slope'] = scf_slopes
chairlifts_pts['elevation'] = elevations

# Drop NaNs (e.g., points outside raster extent or in masked areas)
chairlifts_pts = chairlifts_pts.dropna(subset=['scf_slope', 'elevation'])

# 4. Standard Linear Regression
x_reg = chairlifts_pts['scf_slope']
y_reg = chairlifts_pts['elevation']
z = np.polyfit(x_reg, y_reg, 1)
p = np.poly1d(z)

# 5. Interactive hvplot Visualization
scatter_plot = chairlifts_pts.hvplot.scatter(
    x='scf_slope',
    y='elevation',
    hover_cols=['name', 'scf_slope', 'elevation'],
    alpha=0.6,
    size=40,
    color='blue',
    title='Chairlift Analysis: Linear SCF Trend vs. Elevation',
    xlabel='SCF Change per Decade',
    ylabel='Elevation (m)',
    width=800,
    height=500,
    grid=True
)

# Create the regression line for plotting
x_vals = np.linspace(x_reg.min(), x_reg.max(), 100)
y_vals = p(x_vals)
line_df = pd.DataFrame({'x': x_vals, 'y': y_vals})
reg_line = line_df.hvplot.line(x='x', y='y', color='red', line_width=2, label=f'Linear Fit: y={z[0]:.2f}x + {z[1]:.2f}')

# Combine and display
scatter_plot * reg_line